# Full spectral processing

This notebook is designed to handle the full processing of mass spectra from  raw transients inside bruker .d folders. It performs apodization, fast Fourier transform, peak picking, mass calibration, recalibration and spectral alignment.

In [ ]:
import numpy as np
import pandas as pd
import os
#%matplotlib ipympl
import matplotlib.pyplot as plt
import metabolinks as mtl
from metabolinks import align

import full_processing as fp

Place all the files you want to process in the same folder and provide the exact relative path to that folder. If you wish to process only a selection of the files in the folder you can replace the 'files' list with a list of the names of the files you want to select.

In [ ]:
path_to_files = 'Files_To_Align/BEXIII'

files = [f for f in os.listdir(path_to_files) if f.endswith('.d')]

out_name = 'BEXIII_yeasts'

files

This codeblock handles all the processing at the level of individual spectra.

The available apodization functions are 'none', 'hann', 'hamming', 'blackman', 'blackman-harris', 'kaiser', 'sine', 'bartlett', 'welch', 'tukey', or 'gaussian'. Aliases are also accepted. The F parameter is the position of the window maximum as a fraction of transient length. The name is taken from kilgour and van orden, 2015. but it applies to all functions. F = 0.5 yields a symmetric window, F = 0.0 is only the falling half. The 'beta' parameter is for Kaiser apodization only, values under 4 are recommended. The 'tukey_alpha' parameter applies to tukey apodization and should be between 0 and 1, with 0 being equivalent to no apodization and 1 equivalent to hann apodization. 'std_frc' applies to Gaussian apodization and corresponds to the standard deviation (sigma) as a fraction of the window length.

Zero padding (zp_factor) determines how much to padd the transient with zeroes before FFT. Padding works as a naturaal inteprolation that increases the number of data points in the raw profile spectra. Setting zp_factor to very high values (default) facilitates centroiding and leads to the discovery of more compounds, but it also produces a higher risk of creating spectral artifacts.

The mass range for the analysis should match your acquisition range unless you have a reason to restrict it further.

The signal to noise ratio threshold is applied at the level of centroiding and is one of the most consequential parameters for compound discovery. Higher values will result in a smaller number of centroided peaks, but of higher quality.

The calibrant masses must be the exact m/z values you expect to find in the spectrum. Not the neutral masses.

In [ ]:
specs = {}

# raw spectrum processing parameters
apodization_function = 'sine' 
F = 0.5
beta = np.pi
tukey_alpha = 0.1
std_frac = 0.15

# zero-padding factor
zp_factor = 15

# mass range for the analysis
mz_range = (100, 1500)

# signal to noise ratio threshold for centroiding
signal_to_noise = 3.3

# recalibration
calibrants = ['308.091083', '257.247507', '664.116399'] # GSH, Palmitic, NAD
cal_error = 3 # ppm error tolerance for calibrant matching

for f in files:

    f_path = os.path.join(path_to_files, f)
    f_name = f.split('.')[0]
    print('Processing', f_name)

    raw_spec = fp.raw_spectrum(f_path, apodization_function, F, beta, tukey_alpha, std_frac, zp_factor, mz_range)
    cent_spec = fp.pyopenms_centroiding(raw_spec, snr=signal_to_noise)

    recal_spec = fp.recalibrate(cent_spec, mz_ref=calibrants, tol_ppm=cal_error)

    specs[f_name] = recal_spec

print('---------------------------------------------------------')
print(len(specs), 'spectra processed')

In [ ]:
frames = [pd.DataFrame(data={s: specs[s]['cent_ints']}, index=specs[s]['cent_mzs']) for s in specs]

In [ ]:
ppmtol = 1 # PPM Tolerance
min_samples = 2 # Number of times a compound needs to appear across all samples to be included

aligned_spectra = align(frames, ppmtol, min_samples)

aligned_spectra

In [ ]:
print(list(specs))